# Exercise 1: H₂ Bond-Distance Neural Network


**Target:** Import the libraries and set the random seed.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)


**Target:** Define the H₂ Morse potential.


In [ ]:
D_e = 0.1745
a = 1.0282
r_e = 1.401

def morse_potential(r):
    # TODO 1: Return D_e*(1-exp(-a*(r-r_e)))**2-D_e
    return ...


**Target:** Generate H–H distances and reference energies.


In [ ]:
r = np.linspace(0.6, 5.0, 60)

# TODO 2: Calculate the reference energy for every bond distance.
energy = ...

plt.figure(figsize=(7, 4))
plt.plot(r, energy)
plt.axvline(r_e, color="black", linestyle="--")
plt.xlabel("H-H distance (Bohr)")
plt.ylabel("Energy (Hartree)")
plt.title("H₂ Morse Potential")
plt.grid(alpha=0.3)
plt.show()


**Target:** Create training and test sets.


In [ ]:
test_idx = np.arange(4, len(r), 6)
train_idx = np.array([i for i in range(len(r)) if i not in test_idx])

# TODO 3: Select the training and test values.
r_train, energy_train = ..., ...
r_test, energy_test = ..., ...

plt.figure(figsize=(7, 4))
plt.plot(r, energy, color="gray", label="Reference")
plt.scatter(r_train, energy_train, label="Training")
plt.scatter(r_test, energy_test, marker="x", s=60, label="Test")
plt.xlabel("H-H distance (Bohr)")
plt.ylabel("Energy (Hartree)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


**Target:** Normalize the bond distances and energies.


In [ ]:
# TODO 4: Calculate means and standard deviations from the training set.
r_mean, r_std = ..., ...
energy_mean, energy_std = ..., ...

# TODO 5: Normalize the training and test arrays and reshape them to (N, 1).
X_train = ...
y_train = ...
X_test = ...
y_test = ...


**Target:** Build the neural network.


In [ ]:
def tanh(z):
    return np.tanh(z)

def tanh_derivative(z):
    return 1.0 - np.tanh(z)**2

class SimpleNN:
    def __init__(self, n_hidden=32):
        scale1 = np.sqrt(2.0 / (1 + n_hidden))
        scale2 = np.sqrt(2.0 / (n_hidden + 1))

        self.W1 = np.random.randn(1, n_hidden) * scale1
        self.b1 = np.zeros((1, n_hidden))
        self.W2 = np.random.randn(n_hidden, 1) * scale2
        self.b2 = np.zeros((1, 1))
        self.cache = {}

    def forward(self, X):
        # TODO 6: Calculate z1, a1, and y_pred.
        z1 = ...
        a1 = ...
        y_pred = ...
        self.cache = {"X": X, "z1": z1, "a1": a1}
        return y_pred

    def loss(self, y_pred, y_true):
        # TODO 7: Return the mean squared error.
        return ...

    def backward(self, y_pred, y_true):
        X = self.cache["X"]
        z1 = self.cache["z1"]
        a1 = self.cache["a1"]
        n = len(X)

        # TODO 8: Complete backpropagation.
        d_y = ...
        dW2 = ...
        db2 = ...
        d_z1 = ...
        dW1 = ...
        db1 = ...

        return {"dW1": dW1, "db1": db1, "dW2": dW2, "db2": db2}


**Target:** Train the neural network.


In [ ]:
class Adam:
    def __init__(self, learning_rate=0.003, beta1=0.9, beta2=0.999, epsilon=1e-8):
        self.learning_rate = learning_rate
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.m = {}
        self.v = {}
        self.t = 0

    def step(self, model, gradients):
        self.t += 1

        for name in ["W1", "b1", "W2", "b2"]:
            gradient = gradients["d" + name]
            parameter = getattr(model, name)

            if name not in self.m:
                self.m[name] = np.zeros_like(parameter)
                self.v[name] = np.zeros_like(parameter)

            self.m[name] = self.beta1*self.m[name] + (1-self.beta1)*gradient
            self.v[name] = self.beta2*self.v[name] + (1-self.beta2)*gradient**2

            m_corrected = self.m[name] / (1-self.beta1**self.t)
            v_corrected = self.v[name] / (1-self.beta2**self.t)
            parameter -= self.learning_rate*m_corrected/(np.sqrt(v_corrected) + self.epsilon)


def train(model, X_train, y_train, epochs=6000, learning_rate=0.003):
    optimizer = Adam(learning_rate=learning_rate)
    losses = []

    for epoch in range(epochs):
        y_pred = model.forward(X_train)
        loss = model.loss(y_pred, y_train)
        gradients = model.backward(y_pred, y_train)

        # TODO 9: Use the optimizer to update the model parameters.
        ...

        losses.append(loss)

    return losses


np.random.seed(42)
model = SimpleNN(n_hidden=32)
losses = train(model, X_train, y_train)

plt.figure(figsize=(7, 4))
plt.semilogy(losses)
plt.xlabel("Epoch")
plt.ylabel("MSE loss")
plt.title("Training Loss")
plt.grid(alpha=0.3)
plt.show()


**Target:** Evaluate the model on unseen bond distances.


In [ ]:
# TODO 10: Predict normalized test energies and convert them to Hartree.
test_prediction_n = ...
test_prediction = ...

# TODO 11: Calculate the test RMSE in Hartree and kcal/mol.
rmse_hartree = ...
rmse_kcal = ...

print(f"Test RMSE: {rmse_hartree:.6f} Hartree")
print(f"Test RMSE: {rmse_kcal:.2f} kcal/mol")


**Target:** Plot the learned H₂ potential-energy curve.


In [ ]:
r_dense = np.linspace(0.6, 5.0, 400)

# TODO 12: Normalize r_dense, predict its energies, and denormalize them.
X_dense = ...
energy_prediction = ...

plt.figure(figsize=(8, 5))
plt.plot(r_dense, morse_potential(r_dense), linewidth=2, label="Reference")
plt.plot(r_dense, energy_prediction, "--", linewidth=2, label="Neural network")
plt.scatter(r_train, energy_train, s=25, label="Training data")
plt.xlabel("H-H distance (Bohr)")
plt.ylabel("Energy (Hartree)")
plt.title("H₂ Potential Learned from Bond Distance")
plt.legend()
plt.grid(alpha=0.3)
plt.show()
